# Activity 2: Simple House Price Prediction Using Linear Regression

**Unit:** Unit II – Supervised Learning: Regression  
**Objective:** Build a Linear Regression model to predict a continuous house-value target and evaluate it using regression metrics.

## 1. Dataset selection and understanding

- **Dataset:** California Housing Dataset
- **Source:** Scikit-learn documentation: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html
- **Description:** The dataset contains information about California census-block groups. It includes numerical features such as median income, house age, average rooms, average bedrooms, population, average occupancy, latitude and longitude.
- **Target:** `MedHouseVal` – median house value, expressed in units of **$100,000**.
- **Selected input feature:** `MedInc` – median income of households in the block group.

Median income is a reasonable predictor because areas with higher household incomes often have higher property values. This is an association, not proof that income causes house prices to rise.

**Scope:** This is a deliberately simple, single-feature regression model. It is an educational prediction exercise, not a professional property valuation tool.


In [ ]:
# 2. Import libraries and load the dataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

# Rename the target column to make its meaning explicit
df = df.rename(columns={"MedHouseVal": "HouseValue_100k_USD"})

print("First five records:")
display(df.head())

print("\nDataset shape (rows, columns):", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nData types:")
display(df.dtypes.to_frame("Data type"))


In [ ]:
# 3. Summary statistics, missing values and duplicate records

print("Summary statistics:")
display(df.describe().T)

print("\nMissing values in each column:")
display(df.isnull().sum().to_frame("Missing values"))

duplicate_count = df.duplicated().sum()
print("\nNumber of duplicate rows:", duplicate_count)

# Remove exact duplicate rows if any are present
if duplicate_count > 0:
    df = df.drop_duplicates().copy()
    print("Duplicate rows removed. Updated shape:", df.shape)
else:
    print("No duplicate rows found; no rows needed to be removed.")

# Remove rows with missing values in the selected feature or target, if any
before = len(df)
df = df.dropna(subset=["MedInc", "HouseValue_100k_USD"]).copy()
print("Rows removed due to missing selected values:", before - len(df))

# Basic validity check for selected variables
df = df[(df["MedInc"] >= 0) & (df["HouseValue_100k_USD"] >= 0)].copy()
print("Shape after cleaning:", df.shape)


## 4. Select input feature and target

- **X (input):** `MedInc` – median income.
- **y (target):** `HouseValue_100k_USD` – median house value in units of $100,000.

Median income is selected because purchasing power and local economic conditions can be associated with property values. The dataset also contains other potentially useful features, but this activity uses one feature to keep the Linear Regression model simple and interpretable.


In [ ]:
# 5. Visualise the relationship between median income and house value

plt.figure(figsize=(8, 5))
plt.scatter(df["MedInc"], df["HouseValue_100k_USD"], alpha=0.25)
plt.title("Median Income vs Median House Value")
plt.xlabel("Median income (dataset units)")
plt.ylabel("Median house value (units of $100,000)")
plt.grid(alpha=0.2)
plt.show()


**Interpretation:** The scatter plot generally shows a positive association: census-block groups with higher median incomes tend to have higher median house values. The points are spread out, so income alone does not explain all variation. A visible relationship does not by itself prove causation. The data may also show a ceiling effect at the upper end of the recorded house values.


In [ ]:
# 6. Split the data into training and testing sets (80:20)

X = df[["MedInc"]]
y = df["HouseValue_100k_USD"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Total observations:", len(df))
print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))
print("Training proportion: {:.1%}".format(len(X_train) / len(df)))
print("Testing proportion: {:.1%}".format(len(X_test) / len(df)))


The training set is used to learn the model parameters (slope and intercept). The testing set is kept separate and is used to evaluate predictions on observations the model did not train on. The test set is not used to fit the model.


In [ ]:
# 7. Train the Linear Regression model

model = LinearRegression()
model.fit(X_train, y_train)

intercept = model.intercept_
coefficient = model.coef_[0]

print("Learned intercept:", intercept)
print("Learned coefficient for MedInc:", coefficient)
print(
    "Model equation: Predicted house value (in $100,000 units) = "
    f"{intercept:.4f} + ({coefficient:.4f} × MedInc)"
)


**Meaning of the model parameters:**

- **Slope/coefficient:** The expected change in the predicted median house value, measured in $100,000 units, for a one-unit increase in `MedInc`.
- **Intercept:** The model's predicted house value when `MedInc` is zero. This is a mathematical starting point and may not be meaningful for a typical real-world area.
- The equation describes an estimated linear association; it should not be interpreted as a causal relationship.


In [ ]:
# 8. Predict house values for the TEST set only

y_pred = model.predict(X_test)

comparison = pd.DataFrame({
    "Actual house value ($100,000 units)": y_test.values,
    "Predicted house value ($100,000 units)": y_pred
})
comparison["Actual value (USD)"] = comparison["Actual house value ($100,000 units)"] * 100000
comparison["Predicted value (USD)"] = comparison["Predicted house value ($100,000 units)"] * 100000

print("Sample of actual vs predicted TEST observations:")
display(comparison.head(10).round(3))


Compare the actual and predicted columns. A prediction close to the actual value has a smaller error; a larger gap indicates a less accurate prediction. The table above contains test observations, not training predictions.


In [ ]:
# 9. Evaluate the model on the TEST set

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE : {mae:.4f} (units of $100,000)")
print(f"MSE : {mse:.4f} (squared units of $100,000)")
print(f"RMSE: {rmse:.4f} (units of $100,000)")
print(f"R²  : {r2:.4f}")

print("\nMAE in USD: ${:,.0f}".format(mae * 100000))
print("RMSE in USD: ${:,.0f}".format(rmse * 100000))


### Interpretation of regression metrics

- **MAE (Mean Absolute Error):** Average absolute difference between actual and predicted values. Convert the displayed MAE to dollars by multiplying by 100,000.
- **MSE (Mean Squared Error):** Average squared prediction error. Larger errors receive a stronger penalty because errors are squared.
- **RMSE (Root Mean Squared Error):** Square root of MSE. It is in the same units as the target, so it can be interpreted on the house-value scale.
- **R² (coefficient of determination):** Compares the model with a baseline that predicts the test-set mean. A value closer to 1 indicates that the model explains more variation than this baseline. R² can be negative on test data if the model performs worse than the mean-based baseline.

Use the metric values printed above to describe the model's test-set performance. The error values are in units of $100,000 unless explicitly converted to USD.


In [ ]:
# 10. Plot actual vs predicted test values with a perfect-prediction reference line

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.3, label="Test observations")

lower = min(y_test.min(), y_pred.min())
upper = max(y_test.max(), y_pred.max())
plt.plot([lower, upper], [lower, upper], "r--", label="Perfect prediction")

plt.title("Actual vs Predicted House Values (Test Set)")
plt.xlabel("Actual house value (units of $100,000)")
plt.ylabel("Predicted house value (units of $100,000)")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


**Interpretation:** Points close to the diagonal reference line represent more accurate predictions. Points far from the line indicate larger prediction errors. The amount of scatter around the line shows how much prediction error remains. This plot evaluates the model on the held-out test set.


In [ ]:
# 11. Predict a new example using a value within the training feature range

training_min = X_train["MedInc"].min()
training_max = X_train["MedInc"].max()
training_median = X_train["MedInc"].median()

# Use the training median as a reasonable example input
new_medinc = float(training_median)
new_input = pd.DataFrame({"MedInc": [new_medinc]})
new_prediction_100k = model.predict(new_input)[0]
new_prediction_usd = new_prediction_100k * 100000

print(f"Training MedInc range: {training_min:.3f} to {training_max:.3f}")
print(f"New example MedInc: {new_medinc:.3f}")
print(f"Predicted median house value: {new_prediction_100k:.3f} units of $100,000")
print(f"Predicted median house value in USD: ${new_prediction_usd:,.0f}")


## 12. Conclusion

In this activity, the California Housing Dataset was used to build a single-feature Linear Regression model. `MedInc` was selected as the input feature and `HouseValue_100k_USD` as the continuous numerical target. The data was inspected and cleaned where necessary, then divided into an 80% training set and a 20% testing set. The model was fitted using only the training data and evaluated on the held-out test data.

The coefficient and intercept define the fitted linear equation. MAE, MSE, RMSE and R² provide complementary measures of prediction performance. The actual-versus-predicted plot helps identify how closely the test predictions match the observed values. A new prediction was also generated for a median-income value within the training range.

**Limitations:** House prices depend on many factors, including location, property size, number of bedrooms, condition, local demand, interest rates and other economic conditions. This model uses only median income, so it cannot capture all these influences. The dataset represents California census-block groups and its target is a block-group median value, not the sale price of an individual house. Therefore, the model is suitable for learning regression concepts but should not be used as a real-world valuation service.

## 13. Submission checklist

- [ ] Run all notebook cells from top to bottom in Google Colab or Jupyter Notebook.
- [ ] Confirm that the dataset loads and all tables, metrics and plots are visible.
- [ ] Read the printed coefficient, intercept and evaluation metrics.
- [ ] Save or download the executed notebook (`.ipynb`) with outputs.
- [ ] Add your name, roll number and class details if required by your faculty.
- [ ] Submit the notebook using the method specified by your faculty.

**Dataset source:** https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html
